# Lab 6 — Capstone Fraud Report

**Day 06 · Anomaly Detection · Cisco AI/ML Training**

---

## Learning objectives

1. Compare **four** fraud detection approaches on the same test set.
2. Rank models by **F1 (fraud)**.
3. Save `fraud_detection_report.json` for stakeholders.
4. Write an executive summary balancing precision and recall.

> **Checkpoints:** best model **logistic_regression** · F1 = **0.80** · `fraud_detection_report.json` saved

**Companion script:** `../scripts/lab06_capstone_fraud_report.py`

## Capstone models

| Model | Type | Notes |
|-------|------|-------|
| `majority_baseline` | Dummy | Always predicts legit (Lab 2) |
| `logistic_regression` | Supervised | `class_weight='balanced'` |
| `lof_proximity` | Semi-supervised | Trained on legit only (Lab 4) |
| `random_forest` | Supervised ensemble | 100 trees, balanced (Lab 5) |

Same **random_state=42** split throughout Day 6 — fair comparison.

---

## 1. Load data and split

In [ ]:
%matplotlib inline

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import LocalOutlierFactor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

GH_ROOT = Path.cwd().resolve()
if GH_ROOT.name == "notebooks":
    GH_ROOT = GH_ROOT.parents[2]
elif GH_ROOT.name == "day-06":
    GH_ROOT = GH_ROOT.parents[1]
else:
    for parent in [GH_ROOT, *GH_ROOT.parents]:
        if (parent / "data" / "credit-card" / "credit_card_transactions.csv").is_file():
            GH_ROOT = parent
            break

OUTPUT_DIR = GH_ROOT / "hands-on" / "day-06" / "scripts" / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

NUMERIC_FEATURES = ["amount", "distance_from_home"]
CATEGORICAL_FEATURES = ["merchant_category"]

df = pd.read_csv(GH_ROOT / "data" / "credit-card" / "credit_card_transactions.csv")
X = df[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y = df["is_fraud"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), NUMERIC_FEATURES),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES),
    ]
)

X_train_s = preprocess.fit_transform(X_train)
X_test_s = preprocess.transform(X_test)

print(f"test fraud cases: {int(y_test.sum())} / {len(y_test)}")

---

## 2. Score helper and run all models

In [ ]:
def scores(y_true, y_pred) -> dict[str, float]:
    return {
        "precision": round(precision_score(y_true, y_pred, zero_division=0), 4),
        "recall": round(recall_score(y_true, y_pred, zero_division=0), 4),
        "f1": round(f1_score(y_true, y_pred, zero_division=0), 4),
    }


results: list[dict] = []

# Baseline
dummy_pred = DummyClassifier(strategy="most_frequent").fit(X_train, y_train).predict(X_test)
results.append({"model": "majority_baseline", **scores(y_test, dummy_pred)})

# Logistic regression
lr = Pipeline(
    steps=[
        ("preprocess", ColumnTransformer(
            transformers=[
                ("num", StandardScaler(), NUMERIC_FEATURES),
                ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES),
            ]
        )),
        ("clf", LogisticRegression(max_iter=1000, random_state=42, class_weight="balanced")),
    ]
)
lr.fit(X_train, y_train)
results.append({"model": "logistic_regression", **scores(y_test, lr.predict(X_test))})

# LOF
X_legit_s = preprocess.fit_transform(X_train[y_train == 0])
lof = LocalOutlierFactor(n_neighbors=20, contamination=0.02, novelty=True)
lof.fit(X_legit_s)
lof_pred = np.where(lof.predict(X_test_s) == -1, 1, 0)
results.append({"model": "lof_proximity", **scores(y_test, lof_pred)})

# Random Forest
rf = Pipeline(
    steps=[
        ("preprocess", ColumnTransformer(
            transformers=[
                ("num", StandardScaler(), NUMERIC_FEATURES),
                ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES),
            ]
        )),
        ("clf", RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=42)),
    ]
)
rf.fit(X_train, y_train)
results.append({"model": "random_forest", **scores(y_test, rf.predict(X_test))})

print(f"models evaluated: {len(results)}")

---

## 3. Ranked metrics table

In [ ]:
report_df = pd.DataFrame(results).sort_values("f1", ascending=False)
best = report_df.iloc[0]

print("Lab 6 — Capstone fraud report")
display(report_df)

**logistic_regression** wins on F1 — recall **1.0** with precision **0.67** on this tiny test set.

---

## 4. Visualize F1 by model

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
plot_df = report_df.sort_values("f1", ascending=True)
sns.barplot(data=plot_df, x="f1", y="model", ax=ax, palette="viridis")
ax.set_title("Fraud detection models ranked by F1")
ax.set_xlim(0, 1)
plt.tight_layout()
plt.show()

---

## 5. Save JSON report

In [ ]:
report_path = OUTPUT_DIR / "fraud_detection_report.json"
report_path.write_text(report_df.to_json(orient="records", indent=2), encoding="utf-8")

print(f"best model (F1): {best['model']} (F1={best['f1']})")
print(f"report saved: {report_path.name}")
print(f"full path: {report_path}")

---

## 6. Executive summary (template)

**Sample 3-sentence summary for stakeholders:**

1. We evaluated four fraud detectors on 1,000 credit card transactions (10 known fraud cases).
2. **Logistic regression** with balanced class weights achieved the highest F1 (**0.80**), catching all test fraud cases while limiting false alarms better than LOF.
3. Recommend deploying logistic regression for automated scoring, with LOF as a secondary review queue when recall is the top priority.

*Edit for your business: auto-decline vs human review changes the precision/recall trade-off.*

---

## 7. Day 06 recap

In [ ]:
day06 = pd.DataFrame({
    "lab": ["1 Outliers", "2 Imbalance", "3 Resample", "4 LOF", "5 RF", "6 Capstone"],
    "checkpoint": [
        "10 fraud, mean amt ≈ 234",
        "99:1, baseline F1=0",
        "oversample 8→792",
        "LOF recall=1.0 (standalone)",
        "RF F1≈0.67",
        f"best: {best['model']}",
    ],
})
display(day06)

---

## 8. Course arc (Days 1–6)

In [ ]:
course = pd.DataFrame({
    "day": ["01 Foundations", "02 Regression", "03 Classification", "04 MLOps", "05 Clustering", "06 Fraud"],
    "theme": ["Python & data", "Zomato LR", "Lending Club + SHAP", "KNN + MLflow", "NYSE segments", "Credit card fraud"],
})
display(course)

---

## 9. Checkpoint summary

In [ ]:
assert len(report_df) >= 4
assert report_path.is_file()
assert best["model"] == "logistic_regression"
assert abs(best["f1"] - 0.8) < 0.05
assert int(y_test.sum()) == 2
print("✓ All checkpoint assertions passed")

---

## Reflection questions

1. Would you deploy the F1 winner if false declines cost $500 each?
2. What would you monitor in production after launch?
3. Which Day 6 technique would you combine with logistic regression?

**Previous:** [Lab 5 — Ensemble detector](lab05_ensemble_detector.ipynb)  
**Congratulations — you have completed all six days of Cisco AI/ML hands-on training.**